# uqma on Colab — bare runtime to gate G2

Turn-level uncertainty quantification for tool-using medical AI agents.

Run the cells in order. Everything is idempotent, so re-running after a disconnect is safe.

**Set the runtime to a GPU first:** Runtime → Change runtime type → T4/L4/A100.

## 1. What GPU did we get?

This decides the dtype and the largest model that fits. Two traps it catches:
a **T4 has no bfloat16** (compute capability 7.5; bf16 needs 8.0+), and an 8B model
in 16-bit is ~16 GB, which does not fit a 16 GB card once the KV cache is counted.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

## 2. Repo and dependencies

The package itself needs only `requests`; nothing heavy is imported outside the model server.

In [ ]:
!git clone -q https://github.com/pakhomovee/medical-agent /content/uqma || (cd /content/uqma && git pull -q)
%cd /content/uqma
!pip install -q -e '.[dev]'
!python -m pytest tests/ -q 2>&1 | tail -3

Expect ~156 passed, 13 skipped. The skips are the grader tests — they need the answer
key, which is next.

## 3. The answer key (`refsol.py`)

MedAgentBench ships reference solutions separately, to keep them out of training
crawls. Download it from
[this Box link](https://stanfordmedicine.box.com/s/fizv0unyjgkb1r3a83rfn5p3dc673uho)
and upload it here. It is gitignored and must never be committed.

Without it nothing can be graded: `sol` is null for 9 of the 10 task templates, so the
graders recompute expected answers themselves.

In [ ]:
from google.colab import files
import shutil, pathlib

if not pathlib.Path('data/refsol.py').exists():
    up = files.upload()          # choose refsol.py
    name = next(iter(up))
    shutil.move(name, 'data/refsol.py')
print('refsol.py:', pathlib.Path('data/refsol.py').exists())

## 4. Gate G1 — task structure

No GPU, seconds. Answers whether tasks are parameterised templates (which bounds the
statistical power available to the whole thesis) and whether the graders can be
inverted to a gold action.

In [ ]:
!python scripts/g1_task_structure.py --tasks data/test_data_v2.json \
    --refsol data/refsol.py --out results/g1_v2.json

## 5. FHIR server

Colab has no Docker. Irrelevant: the MedAgentBench image is a Spring Boot HAPI FHIR
war plus a preloaded H2 database, so it runs on a JVM. The bootstrap pulls the image
layers over HTTPS, verifies each against a manifest pinned from Docker Hub, unpacks
them and starts the server.

~1.8 GB down, ~5 GB unpacked, ~90 s to load. Takes a few minutes total.

In [ ]:
!bash scripts/bootstrap_fhir.sh /content/fhir

Expect `GATE G0 [PASS]` with 695 patients / 563,426 observations / 21,991 medication
requests / 124,969 procedures / 74,821 conditions. Different numbers mean the
extraction is incomplete.

## 6. Model server

`gpu_profile` prints the serve command matching this GPU — correct dtype, largest
model that fits, and the flags the sweeps depend on.

In [ ]:
!python scripts/gpu_profile.py

Installing vLLM takes several minutes and may replace Colab's preinstalled torch;
**restart the runtime if it says to**, then re-run from cell 2 (everything is cached).

In [ ]:
!pip install -q vllm

Serve in the background. Startup is 2–5 minutes including the weight download —
the cell polls until the API answers.

In [ ]:
import json, os, subprocess, time, urllib.request

rec = json.loads(subprocess.run(['python','scripts/gpu_profile.py','--json'],
                                capture_output=True, text=True).stdout)
gpu, plan = rec['gpus'][0], rec['recommendation']
assert plan['model'], plan['notes']
max_len = 8192 if gpu['memory_gb'] >= 22 else 4096
print(f"serving {plan['model']} in {plan['dtype']} on {gpu['name']}")

cmd = ['vllm','serve',plan['model'],
       '--served-model-name', plan['model'].split('/')[-1],
       '--dtype', plan['dtype'], '--max-model-len', str(max_len),
       '--enable-prefix-caching','--no-enable-chunked-prefill',
       '--gpu-memory-utilization','0.90']
log = open('/content/vllm.log','w')
subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT,
                 start_new_session=True)

for i in range(120):
    try:
        urllib.request.urlopen('http://localhost:8000/v1/models', timeout=3)
        print(f'up after ~{i*5}s'); break
    except Exception:
        time.sleep(5)
else:
    print(open('/content/vllm.log').read()[-3000:])

## 7. Gate G0 — full

Adds the model server and the logprob determinism measurement. vLLM is not bitwise
deterministic across batch compositions, so the same prompt can return slightly
different logprobs — that would be a noise floor under every estimator. Measured at
exactly 0.000e+00 on an RTX 5090; worth re-checking per GPU.

In [ ]:
!python scripts/g0_environment.py --fhir http://localhost:8080/fhir \
    --base-url http://localhost:8000/v1 --determinism-trials 8 --out results/g0.json

## 8. Gate G2 — model gate

**Passes at action SR ≥ 40% and schema-valid ≥ 80%**, read off the *action* column.
Two published open-weight models score 0.00% on action tasks while scoring 8–39% on
query tasks, and an all-negative label set makes AUROC undefined at exactly the turns
the thesis is about.

Two runs, because Qwen3 emits `<think>` blocks that our prefix dispatch classifies as
INVALID. `--strip-think` fixes that, but it is a **scaffold change**, not a parity fix:
scaffold quality alone moves success on this benchmark by more than 20 points, so the
delta is measured rather than folded in silently.

In [ ]:
!python scripts/g2_model_gate.py --tasks data/test_data_v2.json \
    --backend vllm --base-url http://localhost:8000/v1 \
    --fhir http://localhost:8080/fhir --refsol data/refsol.py \
    --per-template 5 --concurrency 2 --out runs/g2_nothink

In [ ]:
!python scripts/g2_model_gate.py --tasks data/test_data_v2.json \
    --backend vllm --base-url http://localhost:8000/v1 \
    --fhir http://localhost:8080/fhir --refsol data/refsol.py \
    --per-template 5 --concurrency 2 --strip-think --out runs/g2_think

## 9. Keep the results

Colab runtimes are ephemeral: free sessions disconnect after ~90 minutes idle and cap
at ~12 hours. Copy anything you care about to Drive before the runtime dies.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/uqma && cp -r results runs /content/drive/MyDrive/uqma/
!ls -la /content/drive/MyDrive/uqma/results

---

### If the runtime disconnects

Re-run from cell 2. The repo re-clones or pulls, the FHIR bootstrap skips anything
already extracted, and HuggingFace weights are cached. Only `refsol.py` needs
re-uploading, since it is gitignored.

### If free-tier Colab is too small

A T4 caps you at Qwen3-1.7B in float16, which will very likely fail G2 — that is the
gate working, not a bug. Note that a float16 run is not directly comparable to the bf16
runs the proposal specifies (§6.6), because logits are the measurement instrument.
For real sweeps you want an A100 or a local card.